# Code Explainer: Comprehensive Usage Guide

This notebook demonstrates the complete workflow for using Code Explainer, including:
- Basic code explanation
- Batch processing
- Multi-agent consensus explanations
- Retrieval-augmented explanations
- API integration
- Advanced features

## 1. Installation and Setup

In [ ]:
# Install code-explainer package
# Uncomment to install from PyPI
# !pip install code-explainer

# Or install from local development
import sys
sys.path.insert(0, '/path/to/code-explainer')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

## 2. Basic Code Explanation

In [ ]:
from code_explainer import CodeExplainer

# Initialize the explainer
explainer = CodeExplainer(model_name="codet5-base", device="cuda")

# Example code snippets
code_samples = [
    "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "list_comp = [x**2 for x in range(10) if x % 2 == 0]",
    "with open('file.txt') as f:\n    content = f.read()"
]

# Explain each sample
for code in code_samples:
    print(f"Code: {code[:40]}...")
    explanation = explainer.explain_code(code)
    print(f"Explanation: {explanation}\n")

## 3. Batch Processing

In [ ]:
import time

# Prepare batch of code snippets
batch_codes = [
    "x = [1, 2, 3, 4, 5]",
    "result = sum(x) / len(x)",
    "print(f'Average: {result}')",
    "if result > 3:\n    print('High average')",
    "dict_comp = {x: x**2 for x in range(5)}"
]

# Batch explanation with timing
start_time = time.time()
explanations = []

for code in batch_codes:
    explanation = explainer.explain_code(code)
    explanations.append({
        'code': code,
        'explanation': explanation
    })

elapsed_time = time.time() - start_time

# Display results
import pandas as pd
df = pd.DataFrame(explanations)
print(df)
print(f"\nBatch processing time: {elapsed_time:.2f}s for {len(batch_codes)} items")

## 4. Multi-Agent Consensus Explanations

In [ ]:
from code_explainer.multi_agent import MultiAgentOrchestrator

# Initialize multi-agent orchestrator
orchestrator = MultiAgentOrchestrator(num_agents=3)

# Code to explain with consensus
test_code = "def quicksort(arr):\n    return arr if len(arr) <= 1 else quicksort([x for x in arr[1:] if x <= arr[0]]) + [arr[0]] + quicksort([x for x in arr[1:] if x > arr[0]])"

# Generate consensus explanation
result = orchestrator.explain_with_consensus(
    code=test_code,
    explainer_fn=lambda code: explainer.explain_code(code),
    num_explanations=3
)

print(f"Code: {test_code[:50]}...")
print(f"\nNumber of explanations: {len(result.explanations)}")
print(f"Consensus confidence: {result.confidence:.2%}")
print(f"\nConsensus explanation:")
print(result.consensus_explanation)

## 5. Retrieval-Augmented Explanations

In [ ]:
from code_explainer import CodeExplainer

# Initialize with retrieval strategy
explainer_rag = CodeExplainer(
    model_name="codet5-base",
    retriever_model="sentence-transformers/all-MiniLM-L6-v2",
    use_retrieval=True
)

# Add code examples to retrieval database
examples = [
    "def factorial(n):\n    return 1 if n <= 1 else n * factorial(n-1)",
    "def is_prime(n):\n    return all(n % i != 0 for i in range(2, int(n**0.5) + 1))",
    "def merge_sorted(a, b):\n    return sorted(a + b)"
]

for example in examples:
    explainer_rag.retriever.index.add_code(example, "mathematical function")

# Explain with retrieval augmentation
query_code = "def gcd(a, b):\n    return a if b == 0 else gcd(b, a % b)"

print("Retrieving similar code examples...")
similar = explainer_rag.retriever.retrieve(query_code, k=2)
print(f"Found {len(similar)} similar examples")

explanation = explainer_rag.explain_code(query_code, strategy="retrieval-augmented")
print(f"\nExplanation: {explanation}")

## 6. Input Sanitization and Validation

In [ ]:
from code_explainer.input_sanitization import InputSanitizer, InputValidator

# Test valid input
valid_code = "x = 42"
sanitized = InputSanitizer.sanitize_code(valid_code)
print(f"Valid code: {sanitized}")

# Test language validation
for lang in ["Python", "JavaScript", "Java"]:
    sanitized_lang = InputSanitizer.sanitize_language(lang)
    print(f"{lang} -> {sanitized_lang}")

# Test batch validation
batch = ["x = 1", "y = 2", "z = x + y"]
validated = InputValidator.validate_batch_request(batch)
print(f"\nValidated batch size: {len(validated)}")

# Test error handling
try:
    InputSanitizer.sanitize_language("NonexistentLanguage")
except ValueError as e:
    print(f"\nExpected error: {e}")

## 7. Cache Management

In [ ]:
from code_explainer import get_model_cache_info, clear_model_cache
from code_explainer.cache_ttl import TTLCache, CacheTTLConfig
import time

# Check cache status
cache_info = get_model_cache_info()
print(f"Cache Info: {cache_info}")

# Test TTL cache
ttl_cache = TTLCache(ttl_seconds=5)

# Add items
ttl_cache.set("key1", "value1")
ttl_cache.set("key2", "value2")

print(f"Cache size: {ttl_cache.size()}")
print(f"Retrieved key1: {ttl_cache.get('key1')}")

# Wait for expiration
print("\nWaiting for cache to expire...")
time.sleep(6)

print(f"Expired key1: {ttl_cache.get('key1')}")
print(f"Cleanup removed: {ttl_cache.cleanup_expired()} items")

## 8. REST API Integration

In [ ]:
import requests
import json

# API endpoint configuration
BASE_URL = "http://localhost:8000/api/v1"

# Check server health
try:
    response = requests.get(f"{BASE_URL}/health")
    health = response.json()
    print(f"Server Health: {health['status']}")
    print(f"Models Loaded: {health.get('models_loaded', 'N/A')}")
except requests.exceptions.ConnectionError:
    print("Server not available. Start with: uvicorn app:app --reload")

# Single code explanation
try:
    response = requests.post(
        f"{BASE_URL}/explain",
        json={
            "code": "def hello():\n    return 'world'",
            "language": "python",
            "model": "codet5-base",
            "strategy": "vanilla"
        }
    )
    result = response.json()
    print(f"\nExplanation: {result.get('explanation')}")
    print(f"Processing time: {result.get('processing_time')}ms")
except Exception as e:
    print(f"API call failed: {e}")

# Batch explanation
try:
    response = requests.post(
        f"{BASE_URL}/explain/batch",
        json={
            "codes": [
                "x = 1",
                "y = x + 2"
            ],
            "language": "python"
        }
    )
    batch_result = response.json()
    print(f"\nBatch processed: {batch_result.get('total_processed')} items")
except Exception as e:
    print(f"Batch API call failed: {e}")

## 9. Performance Metrics and Logging

In [ ]:
from code_explainer.logging_helpers import get_logger, log_operation, log_error
import time

# Initialize structured logger
logger = get_logger("notebook_example")

# Log operations
log_operation(
    logger,
    "Started code explanation",
    metadata={"model": "codet5-base", "language": "python"}
)

# Simulate work with timing
start_time = time.time()
code = "def example(): pass"
result = explainer.explain_code(code)
elapsed = time.time() - start_time

log_operation(
    logger,
    "Completed code explanation",
    metadata={
        "code_length": len(code),
        "elapsed_seconds": f"{elapsed:.3f}"
    }
)

# Test error logging
try:
    raise ValueError("Example error for logging")
except ValueError as e:
    log_error(
        logger,
        "Processing failed",
        e,
        metadata={"code_sample": code[:20]}
    )

## 10. Advanced Features Summary

In [ ]:
# Feature summary
features = {
    "Basic Explanation": "Single code snippet explanation",
    "Batch Processing": "Multiple code snippets in one call",
    "Multi-Agent Consensus": "Multiple agents for agreement-based explanations",
    "Retrieval Augmented": "Similar code examples to improve explanations",
    "Input Sanitization": "Security validation for code and inputs",
    "Caching": "LRU and TTL-based caching for performance",
    "Logging": "Structured logging with metadata",
    "REST API": "FastAPI integration for HTTP endpoints",
    "Data Governance": "Lineage tracking and retention policies",
    "Quantization": "8-bit quantization for memory efficiency"
}

print("Code Explainer Features:")
print("=" * 50)
for feature, description in features.items():
    print(f"✓ {feature:<20} - {description}")

print("\nNext steps:")
print("1. Customize model selection for your use case")
print("2. Configure cache TTL based on your retention policy")
print("3. Set up API authentication if needed")
print("4. Monitor performance with structured logging")
print("5. Deploy with Docker for production")